# Train SVM & MLP Models for OpenBCI Live Inference

This notebook trains SVM and MLP models on the same OpenBCI data used for the GAT model,
producing models that can be used interchangeably in the live inference dashboard.

All models use the same scaler (from GAT fold1 training) for consistency.

In [1]:
import os, glob, pickle, warnings
import numpy as np
from collections import Counter
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings('ignore')

# ─── Resolve paths ───────────────────────────────────────────────────────────
_cwd = os.path.abspath(os.getcwd())
_candidates = [_cwd] + [os.path.abspath(os.path.join(_cwd, *(['..'] * i))) for i in range(1, 6)]
PROJECT_ROOT = next((p for p in _candidates if os.path.isdir(os.path.join(p, 'data', 'DEAP'))), _cwd)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'recordings_clean', 'data_extracted_v2')
GAT_DIR = os.path.join(PROJECT_ROOT, 'data', 'recordings_clean', 'gat_output_robust')
OUT_DIR = os.path.join(PROJECT_ROOT, 'data', 'recordings_clean', 'openbci_models')
os.makedirs(OUT_DIR, exist_ok=True)

LABEL_MAP = {'calm': [0, 1], 'happy': [1, 1], 'sad': [0, 0], 'stressed': [1, 0]}
N_CH, N_FEATS = 16, 10
SEED = 42

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'DATA_DIR:     {DATA_DIR}')
print(f'OUT_DIR:      {OUT_DIR}')

# ─── Load all data ────────────────────────────────────────────────────────────
print('\nLoading OpenBCI features from data_extracted_v2...')
feat_files = sorted(glob.glob(os.path.join(DATA_DIR, '*_clean_trial_*.npy')))

all_X, all_Y_aro, all_Y_val, all_groups = [], [], [], []
for fp in feat_files:
    fname = os.path.basename(fp)
    cat = fname.split('_clean_trial_')[0]
    if cat not in LABEL_MAP:
        continue
    label = LABEL_MAP[cat]
    arr = np.load(fp, allow_pickle=True)
    for row in arr:
        all_X.append(row[0])
        all_Y_aro.append(label[0])
        all_Y_val.append(label[1])
        all_groups.append(fname)

X = np.array(all_X, dtype=np.float32)  # (N, 16, 10)
Y_aro = np.array(all_Y_aro, dtype=np.int32)
Y_val = np.array(all_Y_val, dtype=np.int32)
groups = np.array(all_groups)

print(f'Total windows: {len(X):,}')
print(f'Unique trials: {len(np.unique(groups))}')
print(f'Arousal dist:  {{0: {(Y_aro==0).sum()}, 1: {(Y_aro==1).sum()}}}')
print(f'Valence dist:  {{0: {(Y_val==0).sum()}, 1: {(Y_val==1).sum()}}}')

# ─── Fit global scaler ────────────────────────────────────────────────────────
print('\nFitting global scaler on all data...')
X_flat = X.reshape(-1, N_CH * N_FEATS)  # (N, 160)
scaler = StandardScaler()
X_scaled_flat = scaler.fit_transform(X_flat)
X_scaled = X_scaled_flat.reshape(-1, N_CH, N_FEATS)

# Save scaler
scaler_path = os.path.join(OUT_DIR, 'scaler.pkl')
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print(f'✓ Scaler saved: {scaler_path}')

# ─── Train SVM (dual-head: separate arousal & valence classifiers) ────────────
print('\n' + '═'*60)
print('TRAINING SVM (RBF, dual-head)')
print('═'*60)

svm_aro = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=SEED)
svm_val = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=SEED)

svm_aro.fit(X_scaled_flat, Y_aro)
print(f'  SVM Arousal trained. Support vectors: {svm_aro.n_support_}')
svm_val.fit(X_scaled_flat, Y_val)
print(f'  SVM Valence trained. Support vectors: {svm_val.n_support_}')

# Evaluate on training data (sanity check)
pred_aro = svm_aro.predict(X_scaled_flat)
pred_val = svm_val.predict(X_scaled_flat)
print(f'  Train F1 Arousal: {f1_score(Y_aro, pred_aro):.4f}')
print(f'  Train F1 Valence: {f1_score(Y_val, pred_val):.4f}')

# Save SVM models
with open(os.path.join(OUT_DIR, 'svm_aro.pkl'), 'wb') as f:
    pickle.dump(svm_aro, f)
with open(os.path.join(OUT_DIR, 'svm_val.pkl'), 'wb') as f:
    pickle.dump(svm_val, f)
print('✓ SVM models saved.')

# ─── Train MLP (dual-head: separate arousal & valence classifiers) ────────────
print('\n' + '═'*60)
print('TRAINING MLP (128→64, dual-head)')
print('═'*60)

mlp_aro = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=300,
                        early_stopping=True, random_state=SEED)
mlp_val = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=300,
                        early_stopping=True, random_state=SEED)

mlp_aro.fit(X_scaled_flat, Y_aro)
print(f'  MLP Arousal trained. Iterations: {mlp_aro.n_iter_}')
mlp_val.fit(X_scaled_flat, Y_val)
print(f'  MLP Valence trained. Iterations: {mlp_val.n_iter_}')

# Evaluate on training data
pred_aro = mlp_aro.predict(X_scaled_flat)
pred_val = mlp_val.predict(X_scaled_flat)
print(f'  Train F1 Arousal: {f1_score(Y_aro, pred_aro):.4f}')
print(f'  Train F1 Valence: {f1_score(Y_val, pred_val):.4f}')

# Save MLP models
with open(os.path.join(OUT_DIR, 'mlp_aro.pkl'), 'wb') as f:
    pickle.dump(mlp_aro, f)
with open(os.path.join(OUT_DIR, 'mlp_val.pkl'), 'wb') as f:
    pickle.dump(mlp_val, f)
print('✓ MLP models saved.')

# ─── Copy GAT model reference ────────────────────────────────────────────────
print('\n' + '═'*60)
print('GAT MODEL')
print('═'*60)
gat_model_path = os.path.join(GAT_DIR, 'best_model.pth')
if os.path.exists(gat_model_path):
    print(f'✓ GAT model exists at: {gat_model_path}')
    # Note: GAT was trained with fold-specific scalers. Using our global scaler
    # will give slightly different (but still reasonable) results for live inference.
else:
    print(f'⚠️ GAT model NOT found at: {gat_model_path}')

# ─── Summary ──────────────────────────────────────────────────────────────────
print('\n' + '═'*60)
print('ALL MODELS READY FOR LIVE INFERENCE')
print('═'*60)
print(f'  GAT:    {gat_model_path}')
print(f'  SVM:    {OUT_DIR}/svm_aro.pkl, svm_val.pkl')
print(f'  MLP:    {OUT_DIR}/mlp_aro.pkl, mlp_val.pkl')
print(f'  Scaler: {scaler_path}')
print('\nDone! Run inference_openbci.ipynb next.')

PROJECT_ROOT: c:\Users\PC\Desktop\EEG_GraphAttentionNetwork
DATA_DIR:     c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\data\recordings_clean\data_extracted_v2
OUT_DIR:      c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\data\recordings_clean\openbci_models

Loading OpenBCI features from data_extracted_v2...
Total windows: 3,388
Unique trials: 94
Arousal dist:  {0: 1596, 1: 1792}
Valence dist:  {0: 1550, 1: 1838}

Fitting global scaler on all data...
✓ Scaler saved: c:\Users\PC\Desktop\EEG_GraphAttentionNetwork\data\recordings_clean\openbci_models\scaler.pkl

════════════════════════════════════════════════════════════
TRAINING SVM (RBF, dual-head)
════════════════════════════════════════════════════════════
  SVM Arousal trained. Support vectors: [375 348]
  SVM Valence trained. Support vectors: [207 198]
  Train F1 Arousal: 0.9966
  Train F1 Valence: 0.9997
✓ SVM models saved.

════════════════════════════════════════════════════════════
TRAINING MLP (128→64, dual-head)
══════════════